# Chapter 03 — N-gram MLP Language Model

The bigram model only looked at one character of context. But language has structure that spans much further — the word 'transformer' only makes sense given several preceding characters. N-gram models extend our context window, and the Multi-Layer Perceptron (MLP) gives us a neural network approach to learning these longer patterns.

### Glossary / Glossário

• n-gram — sequence of n consecutive tokens used as context for prediction. / Sequência de n tokens consecutivos usada como contexto para previsão.
• MLP — Multi-Layer Perceptron, a feedforward neural network with one or more hidden layers. / Perceptron Multi-Camadas, rede neural feedforward com uma ou mais camadas ocultas.
• embedding — dense vector representation of a token that captures semantic similarity. / Representação vetorial densa de um token que captura similaridade semântica.
• matmul — matrix multiplication, the fundamental operation in every neural network layer. / Multiplicação de matrizes, operação fundamental em cada camada de rede neural.
• GELU — Gaussian Error Linear Unit, smooth activation function preferred in modern LLMs. / Gaussian Error Linear Unit, função de ativação suave preferida em LLMs modernos.
• ReLU — Rectified Linear Unit, activation that outputs max(0, x). / Rectified Linear Unit, ativação que retorna max(0, x).
• batch — group of training examples processed together for efficiency and stable gradients. / Grupo de exemplos de treinamento processados juntos para eficiência e gradientes estáveis.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NgramMLP(nn.Module):
    def __init__(self, vocab_size, context_len, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(context_len * embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: (batch, context_len) of token indices
        emb = self.embedding(x)          # (batch, ctx, embed)
        emb = emb.view(emb.size(0), -1)  # (batch, ctx * embed)
        h = F.gelu(self.fc1(emb))        # (batch, hidden)
        logits = self.fc2(h)             # (batch, vocab_size)
        return logits

# Demo: forward pass with synthetic data
model = NgramMLP(
    vocab_size=65,   # number of unique characters in our dataset (a-z, A-Z, digits, punctuation)
    context_len=8,   # how many previous characters the model sees (its "memory window")
    embed_dim=32,    # size of learned vector for each character (captures similarity)
    hidden_dim=128   # width of the hidden layer (model's internal processing capacity)
)
x_batch = torch.randint(0, 65, (4, 8))   # 4 training samples, each with 8 characters of context
y_batch = torch.randint(0, 65, (4,))      # target: the next character to predict

logits = model(x_batch)
loss = F.cross_entropy(logits, y_batch)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input shape:  {tuple(x_batch.shape)}")
print(f"Output shape: {tuple(logits.shape)}")
print(f"Initial loss: {loss.item():.4f} (random ≈ {torch.log(torch.tensor(65.0)).item():.4f})")

# === Training loop: watch the loss drop! ===
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
print(f"\n{'Epoch':>5s} | {'Loss':>8s} | {'vs Random':>10s}")
print("-" * 30)
for epoch in range(10):
    logits = model(x_batch)
    loss = F.cross_entropy(logits, y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 2 == 0:
        print(f"{epoch:>5d} | {loss.item():>8.4f} | {'worse' if loss.item() > 4.17 else 'better'}")
print(f"\nLoss dropped from ~4.17 (random) — the model is learning patterns!")

---

**Course:** Aprenda LLM | **Chapter 03**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.